In [1]:
import pandas as pd
import requests
from tqdm import tqdm  # 导入 tqdm
import requests
import pandas as pd
import time
from typing import Tuple
import json

In [ ]:
# 配置DeepSeek API
API_KEY = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # 替换为您的实际API密钥
API_URL = "https://api.deepseek.com/chat/completions"

In [3]:
def classify_word(word):
    """使用 DeepSeek API 对单个词进行产业细分类，返回大类和小类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # 更新后的详细分类提示词
    prompt = f"""
    你是一个具有专业知识和语义理解能力的经济活动分类助手，任务是对描述村落居民生产活动的词语进行分类。

    你的分类过程分为两个步骤：
    第一步，请你根据语义内容判断该词语最可能属于以下三种**大类**中的哪一类（不要机械套用规则，而是结合实际含义进行判断）：
    1. **农业**：与农作物种植、畜禽养殖、林产品采集、渔业活动、手工副业等第一产业密切相关的词语。例如：种地、施肥、耕牛、鱼塘、木材等。
    2. **工业**：涉及制造、加工、修理、能源、水利工程等第二产业的内容。例如：金属加工、修配厂、水利站、电力设备等。
    3. **其他**：不属于农业或工业范畴的词语，例如服务业、商业、教育、交通、旅游等。

    第二步，在明确大类后，根据以下**小类标准**进一步细分。你需要结合词语含义智能判断最合适的小类，而不是仅仅依靠关键词匹配。分类标准如下：

    一、农业下设：
    - 种植业：粮食作物（如米、麦）、经济作物、肥料、农田、农具、河泥等
    - 畜牧业：养猪、养牛、养羊、养蚕等家畜家禽类活动
    - 林业：树木、林木、药材、果园、木材相关
    - 渔业：捕鱼、养鱼、水产、鱼塘等
    - 副业：农户手工副业，如编织、加工、打猎等
    - 其他：农业中未能明确归类的词语

    二、工业下设：
    - 制造业：产品生产、机械加工、修理、金属制品、设备、伞等
    - 水利：水利工程、灌溉设施、水渠、供水等
    - 电力：发电机、变压器、电流、电线、安培等
    - 其他：工业中无法明确归类的其他内容

    三、其他类不再细分，直接标记为“其他 -> 其他”

    请严格根据上面的流程判断并输出分类结果：
    - 输出格式为：大类 -> 小类（例如：“农业 -> 种植业”、“工业 -> 电力”、“其他 -> 其他”）
    - 请仅输出分类结果，不做解释。

    需要分类的词语：{word}
    """

    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()

        # 提取大类和小类
        if "->" in result:
            main_category, sub_category = result.split("->")
            main_category = main_category.strip()
            sub_category = sub_category.strip()
        else:
            main_category, sub_category = "其他", "其他"

        # 仅保留指定的大类和小类
        main_category = main_category if main_category in ["农业", "工业", "其他"] else "其他"
        sub_category = sub_category if sub_category in ["种植业", "畜牧业", "制造业","林业","渔业","副业","电力","水利"] else "其他"

        return main_category, sub_category

    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "其他", "其他"


def process_excel(file_path):
    """加载 Excel 文件，拆分分词并进行产业分类"""
    # 加载 Excel 文件
    df = pd.read_excel(file_path)

    # 只提取第一列（COL_C）和第六列（segmented_text）
    df_subset = df[['COL_C', 'segmented_text']]

    # 新的列表来保存处理后的数据
    new_rows = []

    # 遍历 DataFrame 中的每一行
    for index, row in df_subset.iterrows():
        col_c_value = row['COL_C']

        # 确保 segmented_text 是字符串类型，并处理 NaN 或其他类型
        segmented_text = str(row['segmented_text']) if pd.notna(row['segmented_text']) else ""

        # 将 segmented_text 按空格拆分成多个词
        words = segmented_text.split(' ') if segmented_text else []

        # 为每个词创建一行新的三元组
        for word in words:
            new_rows.append([col_c_value, word])  # COL_C 和 单个词组合成一行

    # 将新生成的行转换为 DataFrame
    new_df = pd.DataFrame(new_rows, columns=['COL_C', 'Element'])

    # 使用 tqdm 包裹分词列表，显示进度条
    tqdm.pandas(desc="Processing words")  # 设置进度条描述

    # 对每个词进行产业分类
    new_df[['大类', '小类']] = new_df['Element'].progress_apply(lambda x: pd.Series(classify_word(x)))

    # 输出为新的 Excel 文件
    output_path = file_path.replace('.xlsx', '_classified2.0.xlsx')
    new_df.to_excel(output_path, index=False)
    print(f"分类结果已保存到 {output_path}")



In [4]:
# 示例使用
file_path = r"E:\people's commune.xlsx"  # 替换为你的文件路径
process_excel(file_path)

Processing words:  11%|█         | 115/1073 [06:42<1:11:15,  4.46s/it]

分类错误: 农具 - HTTPSConnectionPool(host='api.deepseek.com', port=443): Max retries exceeded with url: /chat/completions (Caused by ProxyError('Cannot connect to proxy.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))


Processing words:  13%|█▎        | 142/1073 [08:14<1:13:54,  4.76s/it]

分类错误: 百合 - HTTPSConnectionPool(host='api.deepseek.com', port=443): Max retries exceeded with url: /chat/completions (Caused by ProxyError('Cannot connect to proxy.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))


Processing words:  16%|█▌        | 173/1073 [10:39<1:27:01,  5.80s/it]

...

Processing words:  91%|█████████ | 975/1073 [54:19<07:30,  4.60s/it]

分类错误: 畜牧 - HTTPSConnectionPool(host='api.deepseek.com', port=443): Max retries exceeded with url: /chat/completions (Caused by ProxyError('Cannot connect to proxy.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))


Processing words: 100%|██████████| 1073/1073 [59:58<00:00,  3.35s/it]

分类结果已保存到 E:\people's commune_classified2.0.xlsx



C:\Users\ruyi\AppData\Local\Temp\ipykernel_23208\3554524004.py:110: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  new_df.to_excel(output_path, index=False)
